# Modern Baseline Benchmarking for Cross-Script Writer Verification

## Research Question

How competitive is the current batch-level adversarial writer-verification model when compared with stronger modern visual representations under the exact same writer-disjoint verification protocol?

## Motivation

The current batch-alternating model consistently improved writer verification and cross-script verification across multiple training seeds. However, its absolute performance cannot be interpreted as state of the art without comparison against strong modern representation baselines evaluated under the same data split and pair protocol.

This notebook therefore performs controlled benchmarking rather than introducing a new method.

## Benchmarking Principles

- Use the existing writer-disjoint QUWI split.
- Use the exact existing validation verification pairs.
- Preserve the same cosine-similarity verification protocol.
- Report ROC-AUC and interpolated EER.
- Report overall, within-script, and cross-script performance.
- Do not use the official test set for model or hyperparameter selection.
- Start with frozen modern representations before introducing adaptation.
- Use development writers only for any learned adaptation such as LDA.
- Compare all methods under the same evaluation protocol.

## Current Reference

The main reference model is the batch-level alternating adversarial model from the previous experiments.

The purpose of this notebook is not to claim state-of-the-art performance immediately, but to determine whether the existing verifier is competitive enough to support the next reliability and uncertainty-aware research phase.

In [9]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

In [2]:
SPLIT_SEED = 42
EMBEDDING_BATCH_SIZE = 16

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "pyproject.toml").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate project root."
    )

IMAGE_DIR = (
    Path.home()
    / "Documents"
    / "Handwriting"
    / "QUWI"
    / "extracted"
    / "images"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

VALIDATION_PAIR_PATH = (
    PROJECT_ROOT
    / "splits"
    / "verification"
    / "quwi_validation_verification_pairs.csv"
)

NOTEBOOK17_REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "multiseed_adversarial_robustness"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "modern_baseline_benchmarking"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_df = pd.read_csv(
    SPLIT_PATH
)

development_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "development_train"
    ]
    .reset_index(drop=True)
)

validation_df = (
    split_df[
        split_df[
            "experiment_split"
        ] == "validation"
    ]
    .reset_index(drop=True)
)

validation_pairs_df = pd.read_csv(
    VALIDATION_PAIR_PATH
)

multiseed_writer_df = pd.read_csv(
    NOTEBOOK17_REPORT_DIR
    / "multiseed_writer_summary.csv"
)

multiseed_cross_df = pd.read_csv(
    NOTEBOOK17_REPORT_DIR
    / "multiseed_cross_script_summary.csv"
)

optional_packages = {
    package: (
        importlib.util.find_spec(
            package
        )
        is not None
    )
    for package in [
        "timm",
        "transformers",
    ]
}

TRAIN_DEVICE = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "Device:",
    TRAIN_DEVICE,
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "Torchvision:",
    torchvision.__version__,
)

print()

print(
    "Development writers:",
    development_df[
        "writer"
    ].nunique(),
)

print(
    "Development images:",
    len(
        development_df
    ),
)

print(
    "Validation writers:",
    validation_df[
        "writer"
    ].nunique(),
)

print(
    "Validation images:",
    len(
        validation_df
    ),
)

print(
    "Validation pairs:",
    len(
        validation_pairs_df
    ),
)

print()

print(
    "Current batch-alt mean AUC:",
    multiseed_writer_df[
        "batch_alt_auc"
    ].mean(),
)

print(
    "Current batch-alt mean cross-script AUC:",
    multiseed_cross_df[
        "batch_cross_auc"
    ].mean(),
)

print()

print(
    "Optional packages:",
    optional_packages,
)

Project root: /home/arijit/Documents/handwriting-cross-script-research
Device: cuda
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130

Development writers: 226
Development images: 904
Validation writers: 56
Validation images: 224
Validation pairs: 18816

Current batch-alt mean AUC: 0.7773429715952037
Current batch-alt mean cross-script AUC: 0.7460860196351268

Optional packages: {'timm': False, 'transformers': False}


In [3]:
VIT_WEIGHTS = (
    torchvision.models
    .ViT_B_16_Weights
    .IMAGENET1K_SWAG_E2E_V1
)

VIT_MODEL_NAME = (
    "vit_b_16_swag_e2e_frozen"
)

VIT_INPUT_SIZE = 384

print(
    "Baseline:",
    VIT_MODEL_NAME,
)

print(
    "Input size:",
    VIT_INPUT_SIZE,
)

print(
    "ImageNet top-1:",
    VIT_WEIGHTS.meta[
        "_metrics"
    ][
        "ImageNet-1K"
    ][
        "acc@1"
    ],
)

print(
    "Parameters:",
    VIT_WEIGHTS.meta[
        "num_params"
    ],
)

print(
    "Weights file size (MB):",
    VIT_WEIGHTS.meta[
        "_file_size"
    ],
)

Baseline: vit_b_16_swag_e2e_frozen
Input size: 384
ImageNet top-1: 85.304
Parameters: 86859496
Weights file size (MB): 331.398


In [4]:
vit_model = (
    torchvision.models.vit_b_16(
        weights=VIT_WEIGHTS
    )
)

vit_model.heads = (
    torch.nn.Identity()
)

vit_model = vit_model.to(
    TRAIN_DEVICE
)

vit_model.eval()

for parameter in (
    vit_model.parameters()
):
    parameter.requires_grad = False

dummy_image = torch.zeros(
    1,
    3,
    VIT_INPUT_SIZE,
    VIT_INPUT_SIZE,
    device=TRAIN_DEVICE,
)

with torch.no_grad():
    dummy_embedding = (
        vit_model(
            dummy_image
        )
    )

print(
    "Model device:",
    next(
        vit_model.parameters()
    ).device,
)

print(
    "Embedding shape:",
    dummy_embedding.shape,
)

print(
    "Embedding dimension:",
    dummy_embedding.shape[
        1
    ],
)

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in (
            vit_model.parameters()
        )
        if parameter.requires_grad
    ),
)

Downloading: "https://download.pytorch.org/models/vit_b_16_swag-9ac1b537.pth" to /home/arijit/.cache/torch/hub/checkpoints/vit_b_16_swag-9ac1b537.pth


100.0%


Model device: cuda:0
Embedding shape: torch.Size([1, 768])
Embedding dimension: 768
Trainable parameters: 0


In [5]:
from handwriting_cross_script_research.dataset import QUWIDataset

VIT_BATCH_SIZE = 8

vit_preprocess = (
    VIT_WEIGHTS.transforms()
)

validation_dataset = QUWIDataset(
    metadata=validation_df,
    image_dir=IMAGE_DIR,
)

validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=VIT_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

sanity_batch = next(
    iter(
        validation_loader
    )
)

sanity_images = sanity_batch[
    "image"
]

sanity_processed = (
    vit_preprocess(
        sanity_images
    )
)

print(
    "Raw batch shape:",
    sanity_images.shape,
)

print(
    "Raw value range:",
    float(
        sanity_images.min()
    ),
    float(
        sanity_images.max()
    ),
)

print(
    "Processed batch shape:",
    sanity_processed.shape,
)

print(
    "Processed value range:",
    float(
        sanity_processed.min()
    ),
    float(
        sanity_processed.max()
    ),
)

print(
    "Validation batches:",
    len(
        validation_loader
    ),
)

Raw batch shape: torch.Size([8, 3, 384, 384])
Raw value range: 0.25882354378700256 1.0
Processed batch shape: torch.Size([8, 3, 384, 384])
Processed value range: -0.9876701831817627 2.640000104904175
Validation batches: 28


In [6]:
def extract_vit_embeddings(
    model,
    loader,
):
    embeddings = []
    filenames = []

    model.eval()

    with torch.no_grad():
        for batch in loader:
            images = (
                batch[
                    "image"
                ]
                .to(
                    TRAIN_DEVICE
                )
            )

            processed_images = (
                vit_preprocess(
                    images
                )
            )

            batch_embeddings = (
                model(
                    processed_images
                )
            )

            batch_embeddings = (
                torch.nn.functional.normalize(
                    batch_embeddings,
                    p=2,
                    dim=1,
                )
            )

            embeddings.append(
                batch_embeddings
                .detach()
                .cpu()
                .numpy()
            )

            filenames.extend(
                list(
                    batch[
                        "filename"
                    ]
                )
            )

    return (
        np.concatenate(
            embeddings,
            axis=0,
        ),
        filenames,
    )


vit_validation_embeddings, vit_validation_filenames = (
    extract_vit_embeddings(
        vit_model,
        validation_loader,
    )
)

print(
    "Validation embeddings:",
    vit_validation_embeddings.shape,
)

print(
    "Validation filenames:",
    len(
        vit_validation_filenames
    ),
)

print(
    "Mean embedding norm:",
    np.linalg.norm(
        vit_validation_embeddings,
        axis=1,
    ).mean(),
)

print(
    "Minimum embedding norm:",
    np.linalg.norm(
        vit_validation_embeddings,
        axis=1,
    ).min(),
)

print(
    "Maximum embedding norm:",
    np.linalg.norm(
        vit_validation_embeddings,
        axis=1,
    ).max(),
)

Validation embeddings: (224, 768)
Validation filenames: 224
Mean embedding norm: 1.0
Minimum embedding norm: 0.9999999
Maximum embedding norm: 1.0000001


In [7]:
def calculate_interpolated_eer(
    labels,
    scores,
):
    fpr, tpr, thresholds = roc_curve(
        labels,
        scores,
    )

    fnr = 1.0 - tpr
    difference = fpr - fnr

    crossing_indices = np.where(
        np.diff(
            np.sign(
                difference
            )
        ) != 0
    )[0]

    if len(
        crossing_indices
    ) == 0:
        nearest_index = np.argmin(
            np.abs(
                difference
            )
        )

        eer = (
            fpr[
                nearest_index
            ]
            + fnr[
                nearest_index
            ]
        ) / 2.0

        threshold = thresholds[
            nearest_index
        ]

        return float(
            eer
        ), float(
            threshold
        )

    index = crossing_indices[
        0
    ]

    x0 = difference[
        index
    ]

    x1 = difference[
        index
        + 1
    ]

    weight = (
        -x0
        / (
            x1
            - x0
        )
    )

    eer = (
        fpr[
            index
        ]
        + weight
        * (
            fpr[
                index
                + 1
            ]
            - fpr[
                index
            ]
        )
    )

    threshold = (
        thresholds[
            index
        ]
        + weight
        * (
            thresholds[
                index
                + 1
            ]
            - thresholds[
                index
            ]
        )
    )

    return float(
        eer
    ), float(
        threshold
    )


vit_embedding_map = dict(
    zip(
        vit_validation_filenames,
        vit_validation_embeddings,
    )
)

vit_embeddings_a = np.stack(
    [
        vit_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

vit_embeddings_b = np.stack(
    [
        vit_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

vit_pair_labels = validation_pairs_df[
    "pair_label"
].to_numpy(
    dtype=np.int64
)

vit_scores = np.sum(
    vit_embeddings_a
    * vit_embeddings_b,
    axis=1,
)

vit_overall_auc = roc_auc_score(
    vit_pair_labels,
    vit_scores,
)

vit_overall_eer, vit_eer_threshold = (
    calculate_interpolated_eer(
        vit_pair_labels,
        vit_scores,
    )
)

current_batch_mean_auc = (
    multiseed_writer_df[
        "batch_alt_auc"
    ].mean()
)

print(
    "Frozen ViT overall AUC:",
    vit_overall_auc,
)

print(
    "Frozen ViT EER (%):",
    100.0
    * vit_overall_eer,
)

print(
    "Frozen ViT EER threshold:",
    vit_eer_threshold,
)

print()

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print(
    "ViT AUC difference:",
    vit_overall_auc
    - current_batch_mean_auc,
)

Frozen ViT overall AUC: 0.6713421684961863
Frozen ViT EER (%): 38.39285714285714
Frozen ViT EER threshold: 0.8899178255928887

Current batch-alt mean AUC: 0.7773429715952037
ViT AUC difference: -0.10600080309901738


In [8]:
CONDITION_BY_PAGE_PAIR = {
    (1, 2): "arabic_variable_same",
    (3, 4): "english_variable_same",
    (1, 3): "cross_variable_variable",
    (1, 4): "cross_variable_same",
    (2, 3): "cross_same_variable",
    (2, 4): "cross_same_same",
}

WITHIN_SCRIPT_CONDITIONS = [
    "arabic_variable_same",
    "english_variable_same",
]

CROSS_SCRIPT_CONDITIONS = [
    "cross_variable_variable",
    "cross_variable_same",
    "cross_same_variable",
    "cross_same_same",
]


def filename_page_id(
    filename,
):
    return int(
        Path(
            filename
        ).stem.split(
            "_"
        )[-1]
    )


vit_pair_df = (
    validation_pairs_df
    .copy()
    .reset_index(drop=True)
)

vit_pair_df[
    "score"
] = vit_scores

vit_pair_df[
    "page_a"
] = vit_pair_df[
    "filename_a"
].map(
    filename_page_id
)

vit_pair_df[
    "page_b"
] = vit_pair_df[
    "filename_b"
].map(
    filename_page_id
)

vit_pair_df[
    "condition"
] = [
    CONDITION_BY_PAGE_PAIR[
        (
            int(
                page_a
            ),
            int(
                page_b
            ),
        )
    ]
    for page_a, page_b in zip(
        vit_pair_df[
            "page_a"
        ],
        vit_pair_df[
            "page_b"
        ],
    )
]

vit_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = (
        vit_pair_df[
            vit_pair_df[
                "condition"
            ] == condition
        ]
    )

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    vit_condition_rows.append(
        {
            "condition": (
                condition
            ),
            "auc": float(
                auc
            ),
            "eer": float(
                eer
            ),
        }
    )


vit_condition_df = pd.DataFrame(
    vit_condition_rows
)

vit_within_macro_auc = (
    vit_condition_df[
        vit_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

vit_cross_macro_auc = (
    vit_condition_df[
        vit_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

current_batch_cross_auc = (
    multiseed_cross_df[
        "batch_cross_auc"
    ].mean()
)

print(
    vit_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen ViT within-script macro AUC:",
    vit_within_macro_auc,
)

print(
    "Frozen ViT cross-script macro AUC:",
    vit_cross_macro_auc,
)

print()

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print(
    "ViT cross-script AUC difference:",
    vit_cross_macro_auc
    - current_batch_cross_auc,
)

              condition      auc      eer
   arabic_variable_same 0.726467 0.332468
  english_variable_same 0.752690 0.333117
cross_variable_variable 0.664564 0.397078
    cross_variable_same 0.596107 0.428571
    cross_same_variable 0.663248 0.392857
        cross_same_same 0.667338 0.375000

Frozen ViT within-script macro AUC: 0.7395785018552875
Frozen ViT cross-script macro AUC: 0.6478142393320965

Current batch-alt mean cross-script AUC: 0.7460860196351268
ViT cross-script AUC difference: -0.0982717803030303


In [10]:
development_dataset = QUWIDataset(
    metadata=development_df,
    image_dir=IMAGE_DIR,
)

development_loader = torch.utils.data.DataLoader(
    development_dataset,
    batch_size=VIT_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

vit_development_embeddings, vit_development_filenames = (
    extract_vit_embeddings(
        vit_model,
        development_loader,
    )
)

development_writer_map = dict(
    zip(
        development_df[
            "filename"
        ],
        development_df[
            "writer"
        ],
    )
)

development_writer_labels = np.array(
    [
        development_writer_map[
            filename
        ]
        for filename in (
            vit_development_filenames
        )
    ]
)

print(
    "Development embeddings:",
    vit_development_embeddings.shape,
)

print(
    "Development filenames:",
    len(
        vit_development_filenames
    ),
)

print(
    "Development writers:",
    np.unique(
        development_writer_labels
    ).size,
)

print(
    "Samples per writer:",
    np.unique(
        np.unique(
            development_writer_labels,
            return_counts=True,
        )[
            1
        ]
    ).tolist(),
)

Development embeddings: (904, 768)
Development filenames: 904
Development writers: 226
Samples per writer: [4]


In [11]:
writer_lda = LinearDiscriminantAnalysis(
    solver="svd"
)

writer_lda.fit(
    vit_development_embeddings,
    development_writer_labels,
)

lda_development_embeddings = (
    writer_lda.transform(
        vit_development_embeddings
    )
)

lda_validation_embeddings = (
    writer_lda.transform(
        vit_validation_embeddings
    )
)

lda_development_embeddings = (
    lda_development_embeddings
    / np.linalg.norm(
        lda_development_embeddings,
        axis=1,
        keepdims=True,
    )
)

lda_validation_embeddings = (
    lda_validation_embeddings
    / np.linalg.norm(
        lda_validation_embeddings,
        axis=1,
        keepdims=True,
    )
)

lda_embedding_map = dict(
    zip(
        vit_validation_filenames,
        lda_validation_embeddings,
    )
)

lda_embeddings_a = np.stack(
    [
        lda_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_a"
            ]
        )
    ]
)

lda_embeddings_b = np.stack(
    [
        lda_embedding_map[
            filename
        ]
        for filename in (
            validation_pairs_df[
                "filename_b"
            ]
        )
    ]
)

lda_scores = np.sum(
    lda_embeddings_a
    * lda_embeddings_b,
    axis=1,
)

lda_overall_auc = roc_auc_score(
    vit_pair_labels,
    lda_scores,
)

lda_overall_eer, lda_eer_threshold = (
    calculate_interpolated_eer(
        vit_pair_labels,
        lda_scores,
    )
)

lda_pair_df = (
    vit_pair_df
    .copy()
)

lda_pair_df[
    "score"
] = lda_scores

lda_condition_rows = []

for condition in (
    WITHIN_SCRIPT_CONDITIONS
    + CROSS_SCRIPT_CONDITIONS
):
    condition_df = (
        lda_pair_df[
            lda_pair_df[
                "condition"
            ] == condition
        ]
    )

    labels = condition_df[
        "pair_label"
    ].to_numpy(
        dtype=np.int64
    )

    scores = condition_df[
        "score"
    ].to_numpy()

    auc = roc_auc_score(
        labels,
        scores,
    )

    eer, _ = calculate_interpolated_eer(
        labels,
        scores,
    )

    lda_condition_rows.append(
        {
            "condition": (
                condition
            ),
            "auc": float(
                auc
            ),
            "eer": float(
                eer
            ),
        }
    )


lda_condition_df = pd.DataFrame(
    lda_condition_rows
)

lda_within_macro_auc = (
    lda_condition_df[
        lda_condition_df[
            "condition"
        ].isin(
            WITHIN_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

lda_cross_macro_auc = (
    lda_condition_df[
        lda_condition_df[
            "condition"
        ].isin(
            CROSS_SCRIPT_CONDITIONS
        )
    ][
        "auc"
    ].mean()
)

print(
    "LDA embedding dimension:",
    lda_validation_embeddings.shape[
        1
    ],
)

print()

print(
    "Frozen ViT overall AUC:",
    vit_overall_auc,
)

print(
    "ViT + writer-LDA overall AUC:",
    lda_overall_auc,
)

print(
    "Current batch-alt mean AUC:",
    current_batch_mean_auc,
)

print()

print(
    "ViT + writer-LDA EER (%):",
    100.0
    * lda_overall_eer,
)

print()

print(
    lda_condition_df
    .round(6)
    .to_string(
        index=False
    )
)

print()

print(
    "Frozen ViT cross-script macro AUC:",
    vit_cross_macro_auc,
)

print(
    "ViT + writer-LDA cross-script macro AUC:",
    lda_cross_macro_auc,
)

print(
    "Current batch-alt mean cross-script AUC:",
    current_batch_cross_auc,
)

print()

print(
    "LDA vs batch-alt overall difference:",
    lda_overall_auc
    - current_batch_mean_auc,
)

print(
    "LDA vs batch-alt cross-script difference:",
    lda_cross_macro_auc
    - current_batch_cross_auc,
)

LDA embedding dimension: 225

Frozen ViT overall AUC: 0.6713421684961863
ViT + writer-LDA overall AUC: 0.7035706233250877
Current batch-alt mean AUC: 0.7773429715952037

ViT + writer-LDA EER (%): 35.0

              condition      auc      eer
   arabic_variable_same 0.769707 0.296753
  english_variable_same 0.830746 0.232792
cross_variable_variable 0.651397 0.388312
    cross_variable_same 0.644191 0.379545
    cross_same_variable 0.640080 0.392857
        cross_same_same 0.687077 0.375000

Frozen ViT cross-script macro AUC: 0.6478142393320965
ViT + writer-LDA cross-script macro AUC: 0.6556861665120594
Current batch-alt mean cross-script AUC: 0.7460860196351268

LDA vs batch-alt overall difference: -0.07377234827011603
LDA vs batch-alt cross-script difference: -0.09039985312306742
